In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

import json
import pickle
import torch

from scipy.stats import wilcoxon

# This script calculate p-value of the best base models with FoMo-0D, DTE-NP, kNN, ICL, DTE-C

24.10. model_24_10: SquareGMM of Simon, feat=70, inflate_full=True, delta=0.005, max_feature_dim=70, max_model_dim=70, prior.gmm.max_num_cluster=70, EPOCHS=50, BATCH_SIZE=6, nlayer=6, nhead=2,

---> BEST MODEL: model_24_10

In [2]:
#Fomo-0D model
default_dir = "../results/adbench/context5000.feat100.R500.inf-fullFalse.LTFalse.gen1tr1True.reuse100.E200.step1000.bs8.lr0.001.emb256.hdim512.nhead4.nlayer4.ndevice1/train_seed=0.transform=quantile.feat_truncation=subsampling"
default_dir_best_loss = "../ckpt/context5000.feat100.R500.inf-fullFalse.LTFalse.gen1tr1True.reuse100.E200.step1000.bs8.lr0.001.emb256.hdim512.nhead4.nlayer4.ndevice1/seed0/best.ckpt"

model_24_10_dir = "../results/adbench/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/train_seed=0.transform=quantile.feat_truncation=subsampling"

model_24_10_train_val_loss_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/train_val_loss.pickle"
model_24_10_best_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/best.ckpt"
model_24_10_best_val_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/best_val.ckpt"
model_24_10_last_dir = "../ckpt/(BESTMODEL)dataSGMM.context5000.feat70.R500.inf-fullTrue.LTFalse.gen1tr1True.reuse100.E50.step1000.bs6.lr0.001.emb256.hdim512.nhead2.nlayer6.ndevice1.numclusters70/seed0/last.ckpt"



### 1. best_model with Fomo-0D

In [3]:
# new files:
model_24_10_result = pd.read_csv('processed_model_24_10_results.csv')  #  #      processed_test_validation_ensemble_13_5.csv
default_fomo_result = pd.read_csv('processed_default_fomo.csv')

model_24_10_auc, default_fomo_auc = [], []
for ds, auc, default in zip(model_24_10_result["dataset"],model_24_10_result["aucroc"], default_fomo_result['aucroc']):
    model_24_10_auc.append(auc)
    default_fomo_auc.append(default)

model_24_10_auc = np.array(model_24_10_auc)
default_fomo_auc = np.array(default_fomo_auc)

# Check differences
diffs = model_24_10_auc - default_fomo_auc
print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# One-sided Wilcoxon test
stat, p_value = wilcoxon(model_24_10_auc, default_fomo_auc, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
print(f"mean value of mode_24_10 vs. fomo-0D: {np.mean(model_24_10_auc)} vs. {np.mean(default_fomo_auc)}")  

Mean improvement: -0.0025581941839531294
Better ratio: 0.3508771929824561
Statistic: 681.0
p-value: 0.7017805280392652
mean value of mode_24_10 vs. fomo-0D: 0.7992558340735147 vs. 0.8018140282574678


The model shows a small average improvement in AUC (+0.017) and outperforms the baseline on 58% of datasets. However, the difference is not statistically significant (Wilcoxon signed-rank test, p = 0.127)

### 2. best_model with DTE-NP

In [4]:
#model_24_10_result = pd.read_csv('processed_model_24_10_results.csv')

# Load file 1
df1 = pd.read_csv("processed_model_24_10_results.csv")

# Load file 2
df2 = pd.read_csv("DTE-NP_semi_AUCROC_all_dataset.csv").drop(columns=['DTE_NP_5','DTE_NP_10','DTE_NP_20','DTE_NP_40','DTE_NP_50'])

# --- Clean 
df2["DTE_NP_avr_clean"] = df2["DTE_NP_avr"].str.split("(").str[0].astype(float) / 100.0

# --- Merge on dataset ---
import re

def clean_name(x):
    x = str(x)
    x = re.sub(r"^\d+_", "", x)  # remove prefix like 10_
    x = x.lower()                # lowercase everything
    x = x.strip()                # remove spaces
    return x
df1["dataset_clean"] = df1["dataset"].apply(clean_name)
df2["dataset_clean"] = df2["dataset"].apply(clean_name)
df = pd.merge(df1, df2, on="dataset_clean")

# Extract paired values
auc1 = np.array(df["aucroc"])
auc2 = np.array(df["DTE_NP_avr_clean"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
print(f"mean value of mode_24_10 vs. DTE-NP: {np.mean(df['aucroc'])} vs. {np.mean(df['DTE_NP_avr_clean'])}")

Mean improvement: -0.01032135890894149
Better ratio: 0.38596491228070173
Statistic: 598.0
p-value: 0.9652749153199018
mean value of mode_24_10 vs. DTE-NP: 0.7992558340735147 vs. 0.8095771929824561


--> DTE-NP performs better, but not significant

### 3. best model kNN_semi

In [8]:

# Load file 1
df1 = pd.read_csv("processed_model_24_10_results.csv")

# Load file 2
df2 = pd.read_csv("kNN_semi_AUCROC_all.csv").drop(columns=['KNN_5','KNN_10','KNN_20','KNN_40','KNN_50'])

# --- Clean KNN_avr column ---
df2["KNN_avr_clean"] = df2["KNN_avr"].str.split("(").str[0].astype(float) / 100.0

# --- Merge on dataset ---
import re

def clean_name(x):
    x = str(x)
    x = re.sub(r"^\d+_", "", x)  # remove prefix like 10_
    x = x.lower()                # lowercase everything
    x = x.strip()                # remove spaces
    return x
df1["dataset_clean"] = df1["dataset"].apply(clean_name)
df2["dataset_clean"] = df2["dataset"].apply(clean_name)
df = pd.merge(df1, df2, on="dataset_clean")

# Extract paired values
auc1 = np.array(df["aucroc"])
auc2 = np.array(df["KNN_avr_clean"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
print(f"mean value of mode_24_10 vs. DTE-NP: {np.mean(df['aucroc'])} vs. {np.mean(df['KNN_avr_clean'])}")

Mean improvement: 0.0031067112664971
Better ratio: 0.42105263157894735
Statistic: 698.0
p-value: 0.8463631246570784
mean value of mode_24_10 vs. DTE-NP: 0.7992558340735147 vs. 0.7961491228070176


### 4. best model with ICL

In [9]:

# Load file 1
df1 = pd.read_csv("processed_model_24_10_results.csv")

# Load file 2
df2 = pd.read_csv("ICL_semi_AUCROC_all_dataset.csv").drop(columns=['ICL_0.1','ICL_0.01','ICL_0.001','ICL_0.0001','ICL_1e-05'])

# --- Clean KNN_avr column ---
df2["ICL_avr_clean"] = df2["ICL_avr"].str.split("(").str[0].astype(float) / 100.0

# --- Merge on dataset ---
import re

def clean_name(x):
    x = str(x)
    x = re.sub(r"^\d+_", "", x)  # remove prefix like 10_
    x = x.lower()                # lowercase everything
    x = x.strip()                # remove spaces
    return x
df1["dataset_clean"] = df1["dataset"].apply(clean_name)
df2["dataset_clean"] = df2["dataset"].apply(clean_name)
df = pd.merge(df1, df2, on="dataset_clean")

# Extract paired values
auc1 = np.array(df["aucroc"])
auc2 = np.array(df["ICL_avr_clean"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
print(f"mean value of mode_24_10 vs. DTE-NP: {np.mean(df['aucroc'])} vs. {np.mean(df['ICL_avr_clean'])}")

Mean improvement: 0.002559342845444484
Better ratio: 0.5087719298245614
Statistic: 822.0
p-value: 0.42239508575740803
mean value of mode_24_10 vs. DTE-NP: 0.7992558340735147 vs. 0.7966964912280702


### 5. best model with DTE-C

In [10]:

# Load file 1
df1 = pd.read_csv("processed_model_24_10_results.csv")

# Load file 2
df2 = pd.read_csv("DTE-C_semi_AUCROC_all_dataset.csv").drop(columns=['DTE_C_5','DTE_C_10','DTE_C_20','DTE_C_40','DTE_C_50'])

# --- Clean 
df2["DTE_C_avr_clean"] = df2["DTE_C_avr"].str.split("(").str[0].astype(float) / 100.0

# --- Merge on dataset ---
import re

def clean_name(x):
    x = str(x)
    x = re.sub(r"^\d+_", "", x)  # remove prefix like 10_
    x = x.lower()                # lowercase everything
    x = x.strip()                # remove spaces
    return x
df1["dataset_clean"] = df1["dataset"].apply(clean_name)
df2["dataset_clean"] = df2["dataset"].apply(clean_name)
df = pd.merge(df1, df2, on="dataset_clean")

# Extract paired values
auc1 = np.array(df["aucroc"])
auc2 = np.array(df["DTE_C_avr_clean"])

# --- Compute metrics ---
diffs = auc1 - auc2

print("Mean improvement:", np.mean(diffs))
print("Better ratio:", np.mean(diffs > 0))

# --- Wilcoxon test (one-sided: is model1 better?) ---
stat, p_value = wilcoxon(auc1, auc2, alternative='greater')

print("Statistic:", stat)
print("p-value:", p_value)
print(f"mean value of mode_24_10 vs. DTE-NP: {np.mean(df['aucroc'])} vs. {np.mean(df['DTE_C_avr_clean'])}")

Mean improvement: 0.09014179898579536
Better ratio: 0.8245614035087719
Statistic: 1474.0
p-value: 1.3410124192355665e-07
mean value of mode_24_10 vs. DTE-NP: 0.7992558340735147 vs. 0.7091140350877193


In [11]:
np.sum(df2["DTE_C_avr_clean"]) / len(df2["DTE_C_avr_clean"])

0.7091140350877193

In [17]:
df

,dataset_x,aucroc,aucpr,f1,dataset_clean,dataset_y,DTE_C_avr,DTE_C_avr_clean
0,10_cover,0.994912,0.774384,0.775862,cover,cover,78.40(0.14),0.7840
1,11_donors,0.999119,0.984495,0.982806,donors,donors,80.17(3.55),0.8017
2,12_fault,0.563585,0.584162,0.566122,fault,fault,55.41(0.63),0.5541
3,13_fraud,0.909840,0.046558,0.065217,fraud,fraud,76.21(1.08),0.7621
4,14_glass,0.994989,0.929268,0.900000,glass,glass,76.95(2.58),0.7695
5,15_Hepatitis,1.000000,1.000000,1.000000,hepatitis,hepatitis,84.42(1.20),0.8442
6,16_http,0.988390,0.383444,0.040816,http,http,89.59(0.09),0.8959
7,17_InternetAds,0.761846,0.614812,0.684783,internetads,internetads,66.79(0.73),0.6679
8,18_Ionosphere,0.937438,0.879533,0.911504,ionosphere,ionosphere,83.01(5.81),0.8301
9,19_landsat,0.610592,0.559162,0.497374,landsat,landsat,50.20(0.98),0.5020
